# data_snapshot_registry_v1

目标：验证 `src/data` 的 canonical snapshot 不只写局部 CSV/JSON，还能发布 data update manifest，并维护 dataset registry 的 current/history。

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "AGENTS.md").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data import build_canonical_snapshot, publish_canonical_snapshot

EXPERIMENT = "data_snapshot_registry_v1"
CARD = ROOT / "research" / EXPERIMENT
DATA_ROOT = CARD / "data"
REGISTRY = DATA_ROOT / "data_registry.json"
CARD.mkdir(parents=True, exist_ok=True)


In [2]:
calendar = pd.DatetimeIndex(pd.to_datetime(["2026-01-02", "2026-01-05", "2026-01-06"]), name="date")
raw_v1 = pd.DataFrame([
    {"date": "2026-01-02", "instrument": "SOXL", "close": 10.0, "volume": 100},
    {"date": "2026-01-03", "instrument": "SOXL", "close": 10.5, "volume": 105},
    {"date": "2026-01-05", "instrument": "SOXL", "close": 12.0, "volume": 120},
])
raw_v2 = pd.DataFrame([
    {"date": "2026-01-02", "instrument": "SOXL", "close": 10.0, "volume": 100},
    {"date": "2026-01-05", "instrument": "SOXL", "close": 12.0, "volume": 120},
    {"date": "2026-01-06", "instrument": "SOXL", "close": 13.0, "volume": 130},
])
snapshot_v1 = build_canonical_snapshot(
    raw_v1,
    name="research_unit_soxl_daily",
    universe=("SOXL",),
    calendar=calendar,
    calendar_path="data/qlib/soxl/calendars/day.txt",
    source_paths=("data/raw/research_unit_v1.csv",),
    start="2026-01-01",
    end="2026-01-05",
    as_of="2026-01-05",
)
snapshot_v2 = build_canonical_snapshot(
    raw_v2,
    name="research_unit_soxl_daily",
    universe=("SOXL",),
    calendar=calendar,
    calendar_path="data/qlib/soxl/calendars/day.txt",
    source_paths=("data/raw/research_unit_v2.csv",),
    start="2026-01-01",
    end="2026-01-06",
    as_of="2026-01-06",
)
manifest_v1 = publish_canonical_snapshot(
    snapshot_v1,
    output_dir=DATA_ROOT / "snapshots" / "research_unit_soxl_daily" / "2026-01-05",
    dataset_key="daily.research_unit_soxl",
    registry_path=REGISTRY,
    published_at="2026-01-05T21:00:00Z",
)
manifest_v2 = publish_canonical_snapshot(
    snapshot_v2,
    output_dir=DATA_ROOT / "snapshots" / "research_unit_soxl_daily" / "2026-01-06",
    dataset_key="daily.research_unit_soxl",
    registry_path=REGISTRY,
    published_at="2026-01-06T21:00:00Z",
)
registry = json.loads(REGISTRY.read_text(encoding="utf-8"))
registry


{'datasets': {'daily.research_unit_soxl': {'current': {'as_of': '2026-01-06',
    'dataset_key': 'daily.research_unit_soxl',
    'dataset_name': 'research_unit_soxl_daily',
    'dataset_schema_hash': 'c0f29fcd3a99fd70fb4a288298f09418573f87de94fc9030a11969db41239a1a',
    'end': '2026-01-06',
    'frame_schema_hash': 'd50304bef3df60b6ae09aa909f0226caf58f6f1b8a4d5b99e89f3074383613cb',
    'manifest_hash': '65852074829ba9d78ca3928c99f6dc0dd149243ba58cf17b96e70b396e3f7e3f',
    'manifest_path': '/Users/rziiiii/dev/workspace/research/data_snapshot_registry_v1/data/snapshots/research_unit_soxl_daily/2026-01-06/data_update_manifest.json',
    'published_at': '2026-01-06T21:00:00Z',
    'row_count': 3,
    'snapshot_path': '/Users/rziiiii/dev/workspace/research/data_snapshot_registry_v1/data/snapshots/research_unit_soxl_daily/2026-01-06',
    'start': '2026-01-02'},
   'history': [{'as_of': '2026-01-05',
     'dataset_key': 'daily.research_unit_soxl',
     'dataset_name': 'research_unit_soxl_d

In [3]:
entry = registry["datasets"]["daily.research_unit_soxl"]
summary = {
    "schema_version": "research.data_snapshot_registry_v1",
    "status": "completed",
    "dataset_key": "daily.research_unit_soxl",
    "registry_schema_version": registry["schema_version"],
    "current_manifest_hash": entry["current"]["manifest_hash"],
    "current_end": entry["current"]["end"],
    "history_count": len(entry["history"]),
    "history_manifest_hashes": [item["manifest_hash"] for item in entry["history"]],
    "v1_manifest_hash": manifest_v1["manifest_hash"],
    "v2_manifest_hash": manifest_v2["manifest_hash"],
    "registry_path": str(REGISTRY.relative_to(CARD)),
}
(CARD / "data_registry_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary


{'schema_version': 'research.data_snapshot_registry_v1',
 'status': 'completed',
 'dataset_key': 'daily.research_unit_soxl',
 'registry_schema_version': 'data.registry.v1',
 'current_manifest_hash': '65852074829ba9d78ca3928c99f6dc0dd149243ba58cf17b96e70b396e3f7e3f',
 'current_end': '2026-01-06',
 'history_count': 2,
 'history_manifest_hashes': ['22275f17b5e58954b6fae688c9ff642909e08536ba092e1bf8abaf0ac8811cec',
  '65852074829ba9d78ca3928c99f6dc0dd149243ba58cf17b96e70b396e3f7e3f'],
 'v1_manifest_hash': '22275f17b5e58954b6fae688c9ff642909e08536ba092e1bf8abaf0ac8811cec',
 'v2_manifest_hash': '65852074829ba9d78ca3928c99f6dc0dd149243ba58cf17b96e70b396e3f7e3f',
 'registry_path': 'data/data_registry.json'}

In [4]:
assert summary["registry_schema_version"] == "data.registry.v1"
assert summary["current_manifest_hash"] == summary["v2_manifest_hash"]
assert summary["history_manifest_hashes"] == [summary["v1_manifest_hash"], summary["v2_manifest_hash"]]
assert summary["current_end"] == "2026-01-06"
assert summary["history_count"] == 2
assert (DATA_ROOT / "snapshots" / "research_unit_soxl_daily" / "2026-01-06" / "data_update_manifest.json").exists()


## 中文结论

- 做了什么：新增并验证 `publish_canonical_snapshot`，把 canonical frame、dataset snapshot、data update manifest 和 data registry 串成稳定数据更新证据链。
- 核心指标：本 notebook 发布两版 `daily.research_unit_soxl`，registry schema 为 `data.registry.v1`，current 指向 2026-01-06 版本，history 保留两版 manifest hash。
- regime 差异：本 notebook 是数据层能力，不做 regime 分析；它为后续 `src/analysis` 的固定窗口和 regime/window 对比提供可追溯数据边界。
- 是否晋升：晋升的是 data snapshot registry 框架能力，不晋升任何交易策略。
- 风险：样例使用合成数据；真实数据更新需要 schedule/provider 接入后写入仓库级 `data/`，并保留 provider 原始路径和 as_of。
- 下一步：把 schedule 的数据更新 job 接到 `publish_canonical_snapshot`，并让 dashboard/preflight 读取 `data_registry.json` 判断数据新鲜度。